# Voice & Multimodal AI

Companion notebook for the [Voice & Multimodal AI lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/13-voice-and-multimodal-ai).

We simulate voice AI latency budgets, implement cosine-based visual search (CLIP proxy), and demonstrate multimodal embedding distance. Pure NumPy.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(7)

## 1 — Voice AI latency budget

A voice AI pipeline chains VAD → STT → LLM → TTS. We model the latency distribution for each stage and compute end-to-end p95.

In [ ]:
# Simulate latency (ms) for each stage based on reported production numbers
n_requests = 10_000

# STT (streaming Whisper): mean 180ms, std 40ms
stt_lat  = rng.normal(180, 40, n_requests).clip(80, 500)
# LLM inference (GPT-4o): mean 280ms, std 80ms
llm_lat  = rng.normal(280, 80, n_requests).clip(100, 800)
# TTS first-chunk (streaming): mean 120ms, std 30ms
tts_lat  = rng.normal(120, 30, n_requests).clip(60, 300)
# Total pipeline latency
total    = stt_lat + llm_lat + tts_lat

for stage, lat in [("STT", stt_lat), ("LLM", llm_lat), ("TTS", tts_lat), ("Total", total)]:
    print(f"{stage:8s}: p50={np.percentile(lat,50):.0f}ms  p95={np.percentile(lat,95):.0f}ms  p99={np.percentile(lat,99):.0f}ms")

print(f"\nFraction of requests under 1 second: {(total < 1000).mean():.1%}")

plt.figure(figsize=(8, 3))
plt.hist(total, bins=60, color='#6366f1', alpha=0.8)
plt.axvline(np.percentile(total, 95), color='#f59e0b', lw=2, label=f'p95={np.percentile(total,95):.0f}ms')
plt.axvline(1000, color='#f43f5e', lw=2, ls='--', label='1s budget')
plt.xlabel('End-to-end latency (ms)'); plt.ylabel('Count')
plt.title('Voice AI Pipeline Latency Distribution')
plt.legend(); plt.tight_layout(); plt.show()

## 2 — Visual search with CLIP embeddings

CLIP encodes images and text into the same embedding space. Visual search is ANN retrieval over pre-encoded catalog image embeddings.

In [ ]:
# Simulate a catalog of 1000 "image embeddings" (CLIP d=512 proxy)
n_catalog, d = 1000, 64   # use d=64 for speed (CLIP is 512)
catalog_emb = rng.normal(size=(n_catalog, d))
catalog_emb /= np.linalg.norm(catalog_emb, axis=1, keepdims=True)

# Assign categories (fashion, home, electronics, food)
categories = ['fashion','home','electronics','food']
item_cats = np.array([categories[i % 4] for i in range(n_catalog)])

# "Query image" - embed it with the same CLIP encoder
# Simulate: a fashion item query
fashion_centroid = catalog_emb[item_cats == 'fashion'].mean(0)
fashion_centroid /= np.linalg.norm(fashion_centroid)
query_noise = rng.normal(0, 0.2, d)
query_emb = fashion_centroid + query_noise
query_emb /= np.linalg.norm(query_emb)

# Cosine similarity search
sims = catalog_emb @ query_emb
top_k = 10
top_ids = np.argsort(-sims)[:top_k]

print(f"Top-{top_k} visual search results:")
cat_counts = {c: 0 for c in categories}
for i in top_ids:
    cat_counts[item_cats[i]] += 1
    print(f"  Item {i:4d} [{item_cats[i]:12s}] sim={sims[i]:.4f}")

print(f"\nCategory distribution in top-{top_k}:")
for cat, cnt in cat_counts.items():
    print(f"  {cat}: {cnt}/{top_k} results")

## 3 — Cross-modal retrieval: text → image

CLIP's joint embedding space allows querying images with text (and vice versa). We simulate this with a category-aware "text encoder".

In [ ]:
def text_query_embedding(query_text, category_centroids):
    """
    Simulate a text encoder that maps text to the image embedding space.
    In real CLIP, this is done by the text transformer.
    """
    # Simple proxy: map category keywords to centroids
    for cat, centroid in category_centroids.items():
        if cat in query_text.lower():
            noise = rng.normal(0, 0.15, len(centroid))
            emb = centroid + noise
            return emb / np.linalg.norm(emb)
    return rng.normal(size=list(category_centroids.values())[0].shape)

# Pre-compute category centroids
centroids = {cat: catalog_emb[item_cats == cat].mean(0) for cat in categories}
for cat in centroids:
    centroids[cat] /= np.linalg.norm(centroids[cat])

text_queries = ["show me fashion items", "find home decor", "electronics on sale"]
for query in text_queries:
    q_emb = text_query_embedding(query, centroids)
    sims = catalog_emb @ q_emb
    top3 = np.argsort(-sims)[:3]
    cats = [item_cats[i] for i in top3]
    print(f"Query: '{query}' → top cats: {cats}")

## ✏️ Your turn

**Exercise.** Implement `cosine_recall_at_k(query_emb, catalog_emb, relevant_ids, k)`:
Given a query embedding and catalog, return the fraction of `relevant_ids` that appear in the top-k ANN results (Recall@K).

This is the primary metric for visual and semantic search quality.

In [ ]:
def cosine_recall_at_k(query_emb, catalog_emb, relevant_ids, k=10):
    """
    query_emb:   (d,) query embedding
    catalog_emb: (N, d) catalog embeddings
    relevant_ids: list of relevant item indices (ground truth)
    k: number of results to retrieve
    Returns: recall = |top_k ∩ relevant| / |relevant|
    """
    # TODO(you): compute cosine similarities, retrieve top-k, compute recall
    return ...

# Test: query is close to the first 20 fashion items
relevant = list(np.where(item_cats == 'fashion')[0][:20])
recall = cosine_recall_at_k(query_emb, catalog_emb, relevant, k=20)
print(f"Recall@20 for fashion query: {recall:.4f}")
print(f"Expected: > 0.3 (query is fashion-like)")

In [ ]:
# Assertion
assert 0.0 <= cosine_recall_at_k(query_emb, catalog_emb, relevant, k=20) <= 1.0
# Perfect case: query exactly equals one of the relevant items
perfect_query = catalog_emb[relevant[0]]
r = cosine_recall_at_k(perfect_query, catalog_emb, [relevant[0]], k=1)
assert abs(r - 1.0) < 1e-6, "Exact query should have Recall@1 = 1.0"
print("✓ cosine_recall_at_k correct")

<details><summary>Solution</summary>

```python
def cosine_recall_at_k(query_emb, catalog_emb, relevant_ids, k=10):
    sims = catalog_emb @ query_emb
    top_k_ids = set(np.argsort(-sims)[:k].tolist())
    hits = len(top_k_ids & set(relevant_ids))
    return hits / len(relevant_ids) if relevant_ids else 0.0
```
</details>